In [26]:
import pandas as pd
import numpy as np

In [27]:
# QC output (wide format from your parser)
qc = pd.read_csv("qc_parsed/qc_value_portfolios_wide.csv")

# Your local/backtest portfolio file
local = pd.read_csv("../results/tables/value_top1000_market_cap_top50_long_only/portfolio_membership_at_T.csv")

print(qc.shape, local.shape)
qc.head()

(239, 52) (11900, 26)


,date,n,ticker_1,ticker_2,ticker_3,ticker_4,ticker_5,ticker_6,ticker_7,ticker_8,...,ticker_41,ticker_42,ticker_43,ticker_44,ticker_45,ticker_46,ticker_47,ticker_48,ticker_49,ticker_50
0,2006-01-31,50,GNW,TYC,ALL,CNA,AIG,CD,HIG,AXS,...,NOC,FITB,WLP,MCY,INTC,FE,FPL,ACGL,USB,RGA
1,2006-02-28,50,F,EXPE,STA,UNM,LTR,COP,CNA,GNW,...,RDN,UTSI,VLO,AFL,XOM,KFT,STC,MO,NI,ALL
2,2006-03-31,50,MET,ACE,COF,T,ALL,BAP,HIG,RE,...,KEY,CVX,AFL,INTC,STT,LPX,PFG,MCK,EXPE,RNR
3,2006-04-28,50,F,WLP,TYC,CTL,T,XRX,AIG,CNA,...,STA,ACE,ITT,AXS,WLK,RF,RLI,FITB,LEE,WBS
4,2006-05-31,50,UNM,CD,AXS,FD,AIG,RCL,ACE,RF,...,NOC,STA,WOR,NC,MTG,F,AMP,AOC,AFG,GNW


In [28]:
def wide_to_dict(df):
    portfolio_dict = {}
    
    ticker_cols = [c for c in df.columns if c.startswith("ticker_")]
    
    for _, row in df.iterrows():
        date = pd.to_datetime(row["date"])
        tickers = set(row[ticker_cols].dropna().values)
        portfolio_dict[date] = tickers
    
    return portfolio_dict


qc_dict = wide_to_dict(qc)

In [29]:
local["signal_date"] = pd.to_datetime(local["signal_date"])

local_dict = (
    local.groupby("signal_date")["ticker"]
    .apply(set)
    .to_dict()
)

In [30]:
qc_dates = set(qc_dict.keys())
local_dates = set(local_dict.keys())

common_dates = sorted(qc_dates & local_dates)

print("QC dates:", len(qc_dates))
print("Local dates:", len(local_dates))
print("Overlap:", len(common_dates))

QC dates: 239
Local dates: 238
Overlap: 237


In [31]:
results = []

for date in common_dates:
    qc_set = qc_dict[date]
    local_set = local_dict[date]
    
    intersection = qc_set & local_set
    union = qc_set | local_set
    
    overlap_count = len(intersection)
    qc_size = len(qc_set)
    local_size = len(local_set)
    
    overlap_ratio_qc = overlap_count / qc_size if qc_size else np.nan
    overlap_ratio_local = overlap_count / local_size if local_size else np.nan
    jaccard = overlap_count / len(union) if union else np.nan
    
    results.append({
        "date": date,
        "qc_size": qc_size,
        "local_size": local_size,
        "overlap": overlap_count,
        "overlap_qc_pct": overlap_ratio_qc,
        "overlap_local_pct": overlap_ratio_local,
        "jaccard": jaccard
    })

comparison_df = pd.DataFrame(results).sort_values("date")

comparison_df.head()

,date,qc_size,local_size,overlap,overlap_qc_pct,overlap_local_pct,jaccard
0,2006-02-28,50,50,13,0.26,0.26,0.149425
1,2006-03-31,50,50,14,0.28,0.28,0.162791
2,2006-04-28,50,50,10,0.20,0.20,0.111111
3,2006-05-31,50,50,14,0.28,0.28,0.162791
4,2006-06-30,50,50,14,0.28,0.28,0.162791


In [32]:
comparison_df.describe()

,date,qc_size,local_size,overlap,overlap_qc_pct,overlap_local_pct,jaccard
count,237,237.000000,237.0,237.000000,237.000000,237.000000,237.000000
mean,2015-12-31 11:26:34.936708864,49.915612,50.0,15.789030,0.316118,0.315781,0.190507
min,2006-02-28 00:00:00,30.000000,50.0,6.000000,0.120000,0.120000,0.063830
25%,2011-01-31 00:00:00,50.000000,50.0,13.000000,0.260000,0.260000,0.149425
50%,2015-12-31 00:00:00,50.000000,50.0,16.000000,0.320000,0.320000,0.190476
75%,2020-11-30 00:00:00,50.000000,50.0,18.000000,0.360000,0.360000,0.219512
max,2025-11-28 00:00:00,50.000000,50.0,29.000000,0.580000,0.580000,0.408451
std,NaN,1.299140,0.0,4.169833,0.082760,0.083397,0.059622


In [33]:
# Expect ~50 if your strategy is consistent
print("QC sizes unique:", comparison_df["qc_size"].unique())
print("Local sizes unique:", comparison_df["local_size"].unique())

QC sizes unique: [50 30]
Local sizes unique: [50]


In [34]:
comparison_df.sort_values("overlap").head(10)

,date,qc_size,local_size,overlap,overlap_qc_pct,overlap_local_pct,jaccard
42,2009-08-31,50,50,6,0.12,0.12,0.063830
227,2025-02-28,30,50,6,0.20,0.12,0.081081
179,2021-01-29,50,50,6,0.12,0.12,0.063830
132,2017-02-28,50,50,6,0.12,0.12,0.063830
92,2013-10-31,50,50,7,0.14,0.14,0.075269
185,2021-07-30,50,50,8,0.16,0.16,0.086957
37,2009-03-31,50,50,8,0.16,0.16,0.086957
68,2011-10-31,50,50,9,0.18,0.18,0.098901
144,2018-02-28,50,50,9,0.18,0.18,0.098901
186,2021-08-31,50,50,9,0.18,0.18,0.098901


In [35]:
date = comparison_df.sort_values("overlap").iloc[0]["date"]

qc_set = qc_dict[date]
local_set = local_dict[date]

print("Date:", date)
print("Only in QC:", sorted(qc_set - local_set)[:20])
print("Only in Local:", sorted(local_set - qc_set)[:20])

Date: 2009-08-31 00:00:00
Only in QC: ['ACE', 'ACGL', 'ADM', 'ALL', 'AN', 'AXS', 'BG', 'C', 'CAH', 'CHFC', 'CMCSA', 'CME', 'COF', 'DOW', 'EXC', 'FCF', 'FNB', 'FPL', 'GE', 'JPM']
Only in Local: ['ABG', 'APOG', 'ASR', 'BBOX', 'BMO', 'CATO', 'CATY', 'CLMT', 'CM', 'DD', 'DGICA', 'EBF', 'FINL', 'FLG', 'GGB', 'GHC', 'GPI', 'GVA', 'ITT', 'KELYA']


In [36]:
comparison_df.to_csv("qc_vs_local_comparison.csv", index=False)